In [1]:
from google import genai
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())


# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

In [4]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [24]:
import json
def search(query):
    """Search the documentation database for relevant results based on a query string.
    
    Args:
        query: The search query to look up in the index
    
    """
    results = index.search(
        query=query,
        num_results=5
    )
    return json.dumps(results)

In [25]:
RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [7]:
question = "How do I create a dashboard in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [9]:
response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=user_prompt,
    config={
        "system_instruction": RAG_INSTRUCTIONS
    }
)
print(response.text)

The provided documentation does not contain instructions on how to create a dashboard in Evidently. It focuses on creating and evaluating an LLM judge using Evidently.


In [27]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

from google.genai import types

search_tool_declaration = {
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

search_tool = types.Tool(
    function_declarations=[search_tool_declaration]
)

In [19]:
response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=question,
    config={
        "system_instruction": instructions,
    }
)
print(response.usage_metadata.prompt_token_count)

56


In [28]:
response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=question,
    config={
        "system_instruction": instructions,
        "tools": [search_tool]
    }
)
print(response.usage_metadata.prompt_token_count)

107


In [49]:
response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            )
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash-lite',
  response_id='nrvqaa-CEJusz7IP1uHw0AY',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=15,
    prompt_token_count=107,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=107
      ),
    ],
    total_token_count=122
  )
)

In [29]:
print(response.candidates[0].content.parts)

[Part(
  function_call=FunctionCall(
    args={
      'query': 'create a dashboard'
    },
    name='search'
  )
)]


In [35]:
tool_call = response.candidates[0].content.parts[0]
tool_call

Part(
  function_call=FunctionCall(
    args={
      'query': 'create a dashboard'
    },
    name='search'
  )
)

In [41]:
arguments = tool_call.function_call.args
arguments

{'query': 'create a dashboard'}

In [44]:
search_results = search(query='create dashboard in Evidently')
search_results

'[{"start": 0, "content": "Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\\n\\n<Check>\\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\\n</Check>\\n\\n## Adding Tabs\\n\\nBy default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them.\\n\\n**To add a Tab**:\\n\\n- Enter \\"Edit\\" mode on the Dashboard (top right corner).\\n- Click the plus sign with \\"add Tab\\" on the left.\\n- To create a custom Tab, select \\"empty\\" and enter a name.\\n\\nTo simplify setup, you can start with pre-built Tabs. These are dashboard templates with preset Panel combinations:\\n\\n![Add Dashboard Tab](/images/dashboard/add_dashboard_tab_v2.gif)\\n\\n**Pre-built Tabs** rely on having related Metrics (or Presets that include the specific Metrics) within the Project. If the necessary data is not available, 

In [48]:
search_results = search(**arguments)
search_results

'[{"start": 0, "content": "Dashboards let you create Panels to visualize evaluation results over time. Note that to be able to populate the panels, you must first add Reports with evaluation results to the Project.\\n\\n<Check>\\n  No-code Dashboards are available in the Evidently Cloud and Enterprise.\\n</Check>\\n\\n## Adding Tabs\\n\\nBy default, new Panels appear on a single Dashboard. You can add multiple Tabs to organize them.\\n\\n**To add a Tab**:\\n\\n- Enter \\"Edit\\" mode on the Dashboard (top right corner).\\n- Click the plus sign with \\"add Tab\\" on the left.\\n- To create a custom Tab, select \\"empty\\" and enter a name.\\n\\nTo simplify setup, you can start with pre-built Tabs. These are dashboard templates with preset Panel combinations:\\n\\n![Add Dashboard Tab](/images/dashboard/add_dashboard_tab_v2.gif)\\n\\n**Pre-built Tabs** rely on having related Metrics (or Presets that include the specific Metrics) within the Project. If the necessary data is not available, 

In [50]:
from google.genai import types

# 1. Get the function call
function_call = response.candidates[0].content.parts[0].function_call

# 2. Execute the actual search
search_results = search(function_call.args['query'])

# 3. Build the conversation to send back
contents = [
    types.Content(role='user', parts=[types.Part(text=user_prompt)]),
    types.Content(role='model', parts=response.candidates[0].content.parts),
    types.Content(role='user', parts=[
        types.Part(
            function_response=types.FunctionResponse(
                name=function_call.name,
                response={'result': json.dumps(search_results)}
            )
        )
    ])
]

# 4. Make the second call
final_response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=contents,
    config={"system_instruction": instructions, "tools": [search_tool]}
)

print(final_response.text)

The `Evidently` library allows you to create dashboards to visualize evaluation results over time. You can add panels to these dashboards to display various metrics, such as text evaluations, column distributions, counters, and plots (line, bar, pie charts). You can also organize your dashboards by adding tabs and customize the appearance and content of each panel.

For more detailed information and examples, you can refer to the following documentation pages:
- "Add dashboard panels (UI)": This page describes how to design your Dashboard with custom Panels using the user interface.
- "Add dashboard panels (API)": This page describes how to design your Dashboard with custom Panels using the Python API.


In [53]:
final_response.usage_metadata.prompt_token_count

9222

In [51]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [55]:
final_response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=contents,
    config={"system_instruction": instructions, "tools": [search_tool],
            "response_mime_type": "application/json",
            "response_json_schema": RAGResponse.model_json_schema()}
)

print(final_response.text)

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': "Function calling with a response mime type: 'application/json' is unsupported", 'status': 'INVALID_ARGUMENT'}}